In [18]:
import pandas as pd

url = "http://byungwan.com/class/Coupon_Assignment2.xlsx"
coupon = pd.read_excel(url)

coupon.head()

,CID,Gender,Age,CClass,Discount,Visits,Email,SMS,MStatus,Purchases,Sales
0,1,Male,25,2,Coupon Not Used,1,Opt-in,Opt-in,Married,1,0.5050
1,2,Female,17,2,Coupon Not Used,1,Opt-in,Opt-in,Married,1,0.3300
2,3,Female,38,2,Coupon Not Used,3,Opt-in,Opt-in,Married,1,0.2550
3,4,Female,36,2,Coupon Not Used,6,Opt-in,Opt-in,Single,3,1.8762
4,5,Female,37,2,Coupon Not Used,7,Opt-in,Opt-in,Single,5,6.0940


In [19]:
coupon = pd.get_dummies(coupon, columns=["Gender", "Discount", "Email", "SMS", "MStatus"], drop_first=True, dtype=int)
coupon.rename(columns={"Discount_Coupon Used":"Discount"}, inplace=True)

coupon.head()

,CID,Age,CClass,Visits,Purchases,Sales,Gender_Male,Discount,Email_Opt-out,SMS_Opt-out,MStatus_Single
0,1,25,2,1,1,0.5050,1,0,0,0,0
1,2,17,2,1,1,0.3300,0,0,0,0,0
2,3,38,2,3,1,0.2550,0,0,0,0,0
3,4,36,2,6,3,1.8762,0,0,0,0,1
4,5,37,2,7,5,6.0940,0,0,0,0,1


In [20]:
coupon["Discount"].value_counts()

Discount
0    927
1     73
Name: count, dtype: int64

## Upsampling

In [21]:
from sklearn.utils import resample

coupon0 = coupon[coupon["Discount"] == 0]
coupon1 = coupon[coupon["Discount"] == 1]

coupon1_up = resample(coupon1, replace=True, n_samples=927, random_state=0)
coupon_up = pd.concat([coupon0, coupon1_up], axis=0)

coupon_up["Discount"].value_counts()

Discount
0    927
1    927
Name: count, dtype: int64

## ANN

In [22]:
coupon_up.head()

,CID,Age,CClass,Visits,Purchases,Sales,Gender_Male,Discount,Email_Opt-out,SMS_Opt-out,MStatus_Single
0,1,25,2,1,1,0.5050,1,0,0,0,0
1,2,17,2,1,1,0.3300,0,0,0,0,0
2,3,38,2,3,1,0.2550,0,0,0,0,0
3,4,36,2,6,3,1.8762,0,0,0,0,1
4,5,37,2,7,5,6.0940,0,0,0,0,1


In [23]:
y = coupon_up.iloc[:, 7]
x = coupon_up.iloc[:, [1,2,3,4,5,6,8,9,10]]

In [24]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
sc_model = sc.fit(x)
sc_x = sc_model.transform(x)

## Single-Hidden-Layer

In [25]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

mean = []
sizes = range(1, 5)

for s in sizes:
    mlp_n = MLPClassifier(activation="relu", hidden_layer_sizes=(s), max_iter=3000, random_state=0)
    scores = cross_val_score(mlp_n, sc_x, y, cv=5)
    m = float(np.mean(scores))
    mean.append(m)

In [26]:
mean

[0.770815181758578, 0.7756654767975523, 0.788607853136155, 0.8107292197858236]

## 3-Hidden-Layer

In [ ]:
import itertools

a = b = c = range(1,5)
abcs = list(itertools.product(a,b,c))
mean = []

for abc in abcs:
    mlp_n = MLPClassifier(activation="relu", hidden_layer_sizes=(abc), max_iter=3000, random_state=0)
    score = cross_val_score(mlp_n, sc_x, y, cv=5)
    m = float(np.mean(score))
    mean.append(m)

In [ ]:
mean

## Summary of the model